# Lightweight Fine-Tuning Project


TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA (Low-Rank Adaptation)

*I choose LoRA because it enables efficient fine-tuning by adapting only a small number of additional parameters while also keeping most of the pre-trained model frozen. This is an efficient way of making fine-tuning faster and less memory-intensive.*

* Model: BERT-base-uncased

*I choose BERT-base-uncased because it has several advantages for this task.*
- Pre-trained for understanding sentences.
- Designed for tasks like classification.
- 12 layers, 768 hidden units, and 110M parameters → lighter than larger models.
- Uncased model removes case sensitivity issues.
- Strong baseline performance.


* Evaluation approach: Hugging Face Trainer's evaluation method with an accuracy metric

*It  allows to measure the baseline performance of the pre-trained model and to compare it with the new fine-tuned version.*
- Metric used: Accuracy (measures the proportion of correct predictions).
- Evaluation method: Hugging Face's Trainer.evaluate().
- When evaluation happens: At the end of each training epoch.


* Fine-tuning dataset: "emotion" dataset from Hugging Face

*After reviewing several datasets, I realized "emotion" was a good choice.*
- Small dataset (~16K samples) → Faster training
-  Multi-class task (6 labels) → More complex than binary sentiment classification
- Useful for real-world applications → Chatbots, sentiment analysis, monitoring
- Efficient for LoRA fine-tuning → Can get high accuracy with minimal updates

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [1]:
# Section 1: Loading and Evaluating a Foundation Model

# Install necessary libraries
!pip install transformers datasets evaluate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Import necessary libraries
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from evaluate import evaluator
import numpy as np

In [3]:
# Load the pre-trained model and tokenizer
model_name = "bert-base-uncased"  # Choose an appropriate pre-trained model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)  # 6 labels for emotion

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Load the dataset
dataset = load_dataset("emotion")

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [5]:
# Preprocess the data
def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

encoded_dataset = dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
# Define the evaluation metric
import evaluate
metric = evaluate.load("accuracy")

In [7]:
# Define the evaluation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [8]:
 # Evaluate the model before fine-tuning
training_args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=64,
)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=encoded_dataset["validation"],  # Use validation set for evaluation
    compute_metrics=compute_metrics,
)

eval_results = trainer.evaluate()
print(f"Evaluation results before fine-tuning: {eval_results}")

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: minerva-rose (minerva-rose-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Evaluation results before fine-tuning: {'eval_loss': 1.8540217876434326, 'eval_model_preparation_time': 0.0051, 'eval_accuracy': 0.089, 'eval_runtime': 54.3468, 'eval_samples_per_second': 36.801, 'eval_steps_per_second': 0.589}


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [9]:
# Section 2: Fine-tuning with PEFT

# Import PEFT library
from peft import LoraConfig, get_peft_model











In [10]:
# Define PEFT configuration
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)


In [11]:
# Wrap the model with PEFT
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 299,526 || all params: 109,786,380 || trainable%: 0.2728


In [12]:
# Fine-tuning with Trainer
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,1.524300,1.380191,0.503000
2,1.200400,1.134703,0.582500
3,1.132200,1.096293,0.603500


TrainOutput(global_step=3000, training_loss=1.3172229410807292, metrics={'train_runtime': 3475.2004, 'train_samples_per_second': 13.812, 'train_steps_per_second': 0.863, 'total_flos': 1.2673951137792e+16, 'train_loss': 1.3172229410807292, 'epoch': 3.0})

In [17]:
# Save the PEFT model
trainer.save_model("./peft_model")

Performing Inference with a PEFT Model
TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [18]:
# Section 3: Performing Inference with a PEFT Model
# Load the saved PEFT model weights
from peft import PeftModel  # Import PeftModel
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
model = PeftModel.from_pretrained(model, "./peft_model")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
# Evaluate the PEFT model
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
)

eval_results = trainer.evaluate()
print(f"Evaluation results after fine-tuning: {eval_results}")

Evaluation results after fine-tuning: {'eval_loss': 1.057421088218689, 'eval_model_preparation_time': 0.0086, 'eval_accuracy': 0.6225, 'eval_runtime': 60.1803, 'eval_samples_per_second': 33.233, 'eval_steps_per_second': 0.532}


Based on the comparison of accuracy, the model is significantly better after fine-tuning.
